In [1]:
# 1-dataset model (HTCas9)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_FnCas12a():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_FnCas12a():
    with open('FnCas12a_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_FnCas12a = load_branch1_data_FnCas12a()
    X1_FnCas12a   = np.asarray(X1_FnCas12a)
    X1 = np.concatenate([X1_FnCas12a], axis=0) 

    rates_FnCas12a = load_reaction_rates_FnCas12a()
    rates_FnCas12a   = np.asarray(rates_FnCas12a)
    rates = np.concatenate([rates_FnCas12a], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen FnCas12a dataset
    np.random.seed(42)
    full_indices_FnCas12a = np.arange(len(rates_FnCas12a))
    selected_indices_FnCas12a = np.random.choice(len(full_indices_FnCas12a), size=len(full_indices_FnCas12a), replace=False)
    unseen_indices_FnCas12a = np.setdiff1d(full_indices_FnCas12a, selected_indices_FnCas12a)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_FnCas12a = Subset(hybrid_dataset, unseen_indices_FnCas12a)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_FnCas12a = Subset(hybrid_dataset, selected_indices_FnCas12a)
    trial_loader = DataLoader(selected_set_FnCas12a, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_1_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt: Spearman = 0.41389423659765867
../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt: Spearman = 0.42724056678870875
../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt: Spearman = 0.4041839877812589
../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt: Spearman = 0.40814049793665735
../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt: Spearman = 0.39183400773194893
../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt: Spearman = 0.39867579894991695
../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt: Spearman = 0.3964482726669028
../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt: Spearman = 0.4191859154919934
../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt: Spearman = 0.3977239478725556
../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt: Spearman = 0.39379209330960285


In [3]:
# 2-dataset model (HTCas9+HT11)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_FnCas12a():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_FnCas12a():
    with open('FnCas12a_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_FnCas12a = load_branch1_data_FnCas12a()
    X1_FnCas12a   = np.asarray(X1_FnCas12a)
    X1 = np.concatenate([X1_FnCas12a], axis=0) 

    rates_FnCas12a = load_reaction_rates_FnCas12a()
    rates_FnCas12a   = np.asarray(rates_FnCas12a)
    rates = np.concatenate([rates_FnCas12a], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen FnCas12a dataset
    np.random.seed(42)
    full_indices_FnCas12a = np.arange(len(rates_FnCas12a))
    selected_indices_FnCas12a = np.random.choice(len(full_indices_FnCas12a), size=len(full_indices_FnCas12a), replace=False)
    unseen_indices_FnCas12a = np.setdiff1d(full_indices_FnCas12a, selected_indices_FnCas12a)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_FnCas12a = Subset(hybrid_dataset, unseen_indices_FnCas12a)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_FnCas12a = Subset(hybrid_dataset, selected_indices_FnCas12a)
    trial_loader = DataLoader(selected_set_FnCas12a, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_2_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt: Spearman = 0.507971274838644
../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt: Spearman = 0.5240895155257009
../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt: Spearman = 0.5095228543636721
../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt: Spearman = 0.5169056980460559
../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt: Spearman = 0.5269031767913074
../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt: Spearman = 0.5277886582489792
../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt: Spearman = 0.5228886335630465
../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt: Spearman = 0.5039533454405688
../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt: Spearman = 0.516417669426292
../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt: Spearman = 0.529318025558643


In [5]:
# 4-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5))

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_FnCas12a():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_FnCas12a():
    with open('FnCas12a_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_FnCas12a = load_branch1_data_FnCas12a()
    X1_FnCas12a   = np.asarray(X1_FnCas12a)
    X1 = np.concatenate([X1_FnCas12a], axis=0) 

    rates_FnCas12a = load_reaction_rates_FnCas12a()
    rates_FnCas12a   = np.asarray(rates_FnCas12a)
    rates = np.concatenate([rates_FnCas12a], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen FnCas12a dataset
    np.random.seed(42)
    full_indices_FnCas12a = np.arange(len(rates_FnCas12a))
    selected_indices_FnCas12a = np.random.choice(len(full_indices_FnCas12a), size=len(full_indices_FnCas12a), replace=False)
    unseen_indices_FnCas12a = np.setdiff1d(full_indices_FnCas12a, selected_indices_FnCas12a)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_FnCas12a = Subset(hybrid_dataset, unseen_indices_FnCas12a)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_FnCas12a = Subset(hybrid_dataset, selected_indices_FnCas12a)
    trial_loader = DataLoader(selected_set_FnCas12a, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_4_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt: Spearman = 0.5130331278496991
../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt: Spearman = 0.5087401645974027
../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt: Spearman = 0.5085258398811983
../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt: Spearman = 0.47631291768999323
../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt: Spearman = 0.5246479081075064
../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt: Spearman = 0.5041305294464872
../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt: Spearman = 0.5102928258478999
../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt: Spearman = 0.5254402590297779
../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt: Spearman = 0.4838473107795307
../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt: Spearman = 0.5210949503255026


In [7]:
# 5-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_FnCas12a():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_FnCas12a():
    with open('FnCas12a_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_FnCas12a = load_branch1_data_FnCas12a()
    X1_FnCas12a   = np.asarray(X1_FnCas12a)
    X1 = np.concatenate([X1_FnCas12a], axis=0) 

    rates_FnCas12a = load_reaction_rates_FnCas12a()
    rates_FnCas12a   = np.asarray(rates_FnCas12a)
    rates = np.concatenate([rates_FnCas12a], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen FnCas12a dataset
    np.random.seed(42)
    full_indices_FnCas12a = np.arange(len(rates_FnCas12a))
    selected_indices_FnCas12a = np.random.choice(len(full_indices_FnCas12a), size=len(full_indices_FnCas12a), replace=False)
    unseen_indices_FnCas12a = np.setdiff1d(full_indices_FnCas12a, selected_indices_FnCas12a)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_FnCas12a = Subset(hybrid_dataset, unseen_indices_FnCas12a)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_FnCas12a = Subset(hybrid_dataset, selected_indices_FnCas12a)
    trial_loader = DataLoader(selected_set_FnCas12a, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_5_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt: Spearman = 0.5076447086109848
../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt: Spearman = 0.4727414270399797
../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt: Spearman = 0.46653875697837316
../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt: Spearman = 0.5021090543397111
../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt: Spearman = 0.4926871633155597
../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt: Spearman = 0.4884771632209682
../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt: Spearman = 0.5074275048972546
../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt: Spearman = 0.48762371390242737
../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt: Spearman = 0.4418399194381888
../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt: Spearman = 0.4859676226341093


In [9]:
# 6-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_FnCas12a():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_FnCas12a():
    with open('FnCas12a_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_FnCas12a = load_branch1_data_FnCas12a()
    X1_FnCas12a   = np.asarray(X1_FnCas12a)
    X1 = np.concatenate([X1_FnCas12a], axis=0) 

    rates_FnCas12a = load_reaction_rates_FnCas12a()
    rates_FnCas12a   = np.asarray(rates_FnCas12a)
    rates = np.concatenate([rates_FnCas12a], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen FnCas12a dataset
    np.random.seed(42)
    full_indices_FnCas12a = np.arange(len(rates_FnCas12a))
    selected_indices_FnCas12a = np.random.choice(len(full_indices_FnCas12a), size=len(full_indices_FnCas12a), replace=False)
    unseen_indices_FnCas12a = np.setdiff1d(full_indices_FnCas12a, selected_indices_FnCas12a)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_FnCas12a = Subset(hybrid_dataset, unseen_indices_FnCas12a)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_FnCas12a = Subset(hybrid_dataset, selected_indices_FnCas12a)
    trial_loader = DataLoader(selected_set_FnCas12a, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_6_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt: Spearman = 0.4869786443831843
../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt: Spearman = 0.4960841856230119
../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt: Spearman = 0.4945163808177449
../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt: Spearman = 0.49487058038588405
../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt: Spearman = 0.49451959252358985
../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt: Spearman = 0.5099589167179491
../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt: Spearman = 0.5147710223484556
../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt: Spearman = 0.4929983742145369
../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt: Spearman = 0.4941095770551997
../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt: Spearman = 0.46288093372549394


In [11]:
# 7-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER+iMeta)

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_FnCas12a():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_FnCas12a():
    with open('FnCas12a_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_FnCas12a = load_branch1_data_FnCas12a()
    X1_FnCas12a   = np.asarray(X1_FnCas12a)
    X1 = np.concatenate([X1_FnCas12a], axis=0) 

    rates_FnCas12a = load_reaction_rates_FnCas12a()
    rates_FnCas12a   = np.asarray(rates_FnCas12a)
    rates = np.concatenate([rates_FnCas12a], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen FnCas12a dataset
    np.random.seed(42)
    full_indices_FnCas12a = np.arange(len(rates_FnCas12a))
    selected_indices_FnCas12a = np.random.choice(len(full_indices_FnCas12a), size=len(full_indices_FnCas12a), replace=False)
    unseen_indices_FnCas12a = np.setdiff1d(full_indices_FnCas12a, selected_indices_FnCas12a)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_FnCas12a = Subset(hybrid_dataset, unseen_indices_FnCas12a)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_FnCas12a = Subset(hybrid_dataset, selected_indices_FnCas12a)
    trial_loader = DataLoader(selected_set_FnCas12a, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_7_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt: Spearman = 0.5050001217356129
../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt: Spearman = 0.5039188685044335
../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt: Spearman = 0.4927691753870479
../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt: Spearman = 0.491844863193922
../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt: Spearman = 0.49095219024585085
../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt: Spearman = 0.46219976917213035
../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt: Spearman = 0.458289841948927
../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt: Spearman = 0.5010443450878019
../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt: Spearman = 0.49584698441391045
../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt: Spearman = 0.4954009653662274
